In [19]:
import pandas as pd
import random
import collections
import warnings
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import sys 
from pathlib import Path
from sklearn.model_selection import train_test_split

SEED = 42


# Get project root (one level up from notebooks/)
PROJECT_ROOT = Path("..").resolve()

# Add project root to Python path
sys.path.append(str(PROJECT_ROOT))

# Now import
from env_config import AUDIO_ROOT

SEED   = 0
SPLITS = ["train", "val", "test"]

random.seed(SEED)

In [20]:
df = pd.read_csv("../data/metadata/raga_20_dataset.csv")
df.head()

,track_id,artist,raga,tradition,relative_part
0,Abhishek_Raghuram.Nidhi_Chala_Sukhama,Abhishek_Raghuram,Kalyāṇi,carnatic,RagaDataset/Carnatic/audio/bf4662d4-25c3-4cad-...
1,Abhishek_Raghuram.Isha_Paahimam,Abhishek_Raghuram,Kalyāṇi,carnatic,RagaDataset/Carnatic/audio/bf4662d4-25c3-4cad-...
2,Sanjay_Subrahmanyan.Vanajakshi_-_Varnam,Sanjay_Subrahmanyan,Kalyāṇi,carnatic,RagaDataset/Carnatic/audio/bf4662d4-25c3-4cad-...
3,Sanjay_Subrahmanyan.Paarengum,Sanjay_Subrahmanyan,Kalyāṇi,carnatic,RagaDataset/Carnatic/audio/bf4662d4-25c3-4cad-...
4,Sanjay_Subrahmanyan.Ambarame_Tannire,Sanjay_Subrahmanyan,Kalyāṇi,carnatic,RagaDataset/Carnatic/audio/bf4662d4-25c3-4cad-...


In [21]:
df["audio_path"] = df["relative_part"].apply(lambda x: AUDIO_ROOT / x)
df.head()

,track_id,artist,raga,tradition,relative_part,audio_path
0,Abhishek_Raghuram.Nidhi_Chala_Sukhama,Abhishek_Raghuram,Kalyāṇi,carnatic,RagaDataset/Carnatic/audio/bf4662d4-25c3-4cad-...,C:\Users\Sragv\MIR Carnatic raga identificatio...
1,Abhishek_Raghuram.Isha_Paahimam,Abhishek_Raghuram,Kalyāṇi,carnatic,RagaDataset/Carnatic/audio/bf4662d4-25c3-4cad-...,C:\Users\Sragv\MIR Carnatic raga identificatio...
2,Sanjay_Subrahmanyan.Vanajakshi_-_Varnam,Sanjay_Subrahmanyan,Kalyāṇi,carnatic,RagaDataset/Carnatic/audio/bf4662d4-25c3-4cad-...,C:\Users\Sragv\MIR Carnatic raga identificatio...
3,Sanjay_Subrahmanyan.Paarengum,Sanjay_Subrahmanyan,Kalyāṇi,carnatic,RagaDataset/Carnatic/audio/bf4662d4-25c3-4cad-...,C:\Users\Sragv\MIR Carnatic raga identificatio...
4,Sanjay_Subrahmanyan.Ambarame_Tannire,Sanjay_Subrahmanyan,Kalyāṇi,carnatic,RagaDataset/Carnatic/audio/bf4662d4-25c3-4cad-...,C:\Users\Sragv\MIR Carnatic raga identificatio...


In [22]:
df["raga"].value_counts()


raga
Kalyāṇi            12
Kāmavardani        12
Ṣanmukhapriya      12
Dhanyāsi           12
Tōḍi               12
Sahānā             12
Bilahari           12
Bhairavi           12
Mōhanaṁ            12
Karaharapriya      12
Rītigauḷa          12
Kāṁbhōji           12
Harikāmbhōji       12
Nāṭakurinji        12
Madhyamāvati       12
Śankarābharaṇaṁ    12
Ānandabhairavi     12
Māyāmāḷavagauḷa    12
Kāpi               12
Bēgaḍa             11
Name: count, dtype: int64

In [23]:
df.isnull().sum()

track_id         0
artist           0
raga             0
tradition        0
relative_part    0
audio_path       0
dtype: int64

In [24]:
df['track_id'].duplicated().sum()

0

In [25]:
import os

def fix_long_path(p):
    if os.name == "nt":
        p = os.path.normpath(p)  # convert / → \ properly
        
        if len(p) >= 260 and not p.startswith("\\\\?\\"):
            return "\\\\?\\" + p
    return p

df['audio_path'] = df['audio_path'].apply(fix_long_path)

In [26]:
import os

# Check if all audio paths exist
missing_paths = []
bad = []
for path in df['audio_path']:
    if not os.path.exists(path):
        missing_paths.append(path)
    
long_paths = [p for p in df['audio_path'] if len(p) > 260]
print(len(long_paths))

print(f"Total audio files: {len(df)}")
print(f"Missing audio files: {len(missing_paths)}")

if missing_paths:
    print("\nFirst few missing paths:")
    for path in missing_paths[:5]:
        print(f"  - {path}")
else:
    print("\nAll audio paths exist!")

1
Total audio files: 239
Missing audio files: 4

First few missing paths:
  - C:\Users\Sragv\MIR Carnatic raga identification\data\raw_audio\RagaDataset\Carnatic\audio\bf4662d4-25c3-4cad-9249-4de2dc513c06\Tanjore_S_Kalyanaraman\Tanjore_S_KALYANARAMAN_Live_in_concert-1972-vol_I_and_vol_II\Talli_Ninnu_Nera\Talli_Ninnu_Nera.mp3
  - C:\Users\Sragv\MIR Carnatic raga identification\data\raw_audio\RagaDataset\Carnatic\audio\18b1acb9-dff6-47ec-873a-b2086c8d268d\Cherthala_Ranganatha_Sharma\Cherthala_Ranganatha_Sharma_at_Arkay\Shlokam_-_Shivah_Shaktyayukto\Shlokam_-_Shivah_Shaktyayukto.mp3
  - C:\Users\Sragv\MIR Carnatic raga identification\data\raw_audio\RagaDataset\Carnatic\audio\5ce23030-f71d-4f5d-9d76-f91c2c182392\Gayathri_Venkataraghavan\December_Season_2005\Kundram_Endi_-_Balakrishnan_Pada_Malar\Kundram_Endi_-_Balakrishnan_Pada_Malar.mp3
  - \\?\C:\Users\Sragv\MIR Carnatic raga identification\data\raw_audio\RagaDataset\Carnatic\audio\0277eae5-3411-4b22-9fa8-1b347e7528d1\Prasanna_Venkataram

In [27]:
df = df[df['audio_path'].apply(os.path.exists)].reset_index(drop=True)
print(len(df))

235


We will split the data based on artists so that there is no leakage of information.  
We assign an entire artist to train/ test / val set. 


In [28]:
# Step 1: 80% train+val | 20% test
train_val, test = train_test_split(
    df,
    test_size=0.20,
    stratify=df["raga"],
    random_state=SEED
)

# Step 2: 70% train | 10% val  (10/80 = 0.125 of the remainder)
train, val = train_test_split(
    train_val,
    test_size=0.125,
    stratify=train_val["raga"],
    random_state=SEED
)

# Tag each split
train = train.copy(); train["split"] = "train"
val   = val.copy();   val["split"]   = "val"
test  = test.copy();  test["split"]  = "test"

df = pd.concat([train, val, test]).sort_index()

print(f"train : {len(train):>5,}  ({len(train)/len(df)*100:.1f}%)")
print(f"val   : {len(val):>5,}  ({len(val)/len(df)*100:.1f}%)")
print(f"test  : {len(test):>5,}  ({len(test)/len(df)*100:.1f}%)")

train :   164  (69.8%)
val   :    24  (10.2%)
test  :    47  (20.0%)


In [29]:
# Raga x split distribution
coverage = (
    df.groupby(["raga", "split"])
    .size()
    .unstack(fill_value=0)[["train", "val", "test"]]
)
print(coverage.to_string())

# Check for any raga missing in any split
missing = coverage[(coverage == 0).any(axis=1)]
if missing.empty:
    print("\nAll ragas present in every split.")
else:
    print(f"\nWARNING — ragas missing in a split:\n{missing}")

split            train  val  test
raga                             
Bhairavi             8    2     2
Bilahari             8    2     2
Bēgaḍa               8    1     2
Dhanyāsi             8    1     2
Harikāmbhōji         8    2     2
Kalyāṇi              8    1     2
Karaharapriya        8    1     3
Kāmavardani          9    1     2
Kāpi                 8    1     3
Kāṁbhōji             8    2     2
Madhyamāvati         8    1     2
Māyāmāḷavagauḷa      8    1     3
Mōhanaṁ              9    1     2
Nāṭakurinji          8    1     3
Rītigauḷa            9    1     2
Sahānā               8    1     3
Tōḍi                 8    1     3
Ānandabhairavi       9    1     2
Śankarābharaṇaṁ      8    1     3
Ṣanmukhapriya        8    1     2

All ragas present in every split.


In [30]:
# Summary
summary = (
    df.groupby("split")
    .agg(tracks=("raga", "count"), artists=("artist", "nunique"), ragas=("raga", "nunique"))
    .reindex(["train", "val", "test"])
)
summary["track_%"] = (summary["tracks"] / summary["tracks"].sum() * 100).round(1)
print(summary)

       tracks  artists  ragas  track_%
split                                 
train     164       41     20     69.8
val        24       15     20     10.2
test       47       25     20     20.0


In [ ]:
df.to_csv("../data/metadata/raga_20_dataset_frozen.csv")